In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
import time

#xgboost
from xgboost import XGBClassifier

# sklearn
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, KFold, cross_validate, cross_val_score, RandomizedSearchCV, StratifiedKFold


from sklearn.metrics import classification_report, confusion_matrix, accuracy_score,roc_auc_score
from sklearn.base import BaseEstimator, TransformerMixin, clone

from scipy.stats import ttest_rel
from scipy.stats import randint, uniform

#hiperparamentros search
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier
from scipy.stats import randint, uniform,loguniform


# Importações locais
from setup_notebook import setup_path
setup_path()
from src.model_utils import *
from src.preprocess_utils_diab2 import *
#from src.preprocess_utils_diab2 import *


BASE = Path.cwd().parent
DATA_DIR = BASE/"data"/"raw"
X       = pd.read_csv(DATA_DIR/"train.csv")
X_train = pd.read_csv(DATA_DIR/"X_train_raw.csv")
X_test  = pd.read_csv(DATA_DIR/"X_test_raw.csv")
y_train = pd.read_csv(DATA_DIR/"y_train_raw.csv").values.ravel()
y_test  = pd.read_csv(DATA_DIR/"y_test_raw.csv").values.ravel()
print("\n#Processo finalizado em:", time.strftime("%H:%M:%S"))



#Processo finalizado em: 15:18:27


In [2]:
#função basica
def vizualizar_transformacao(PP,X):
    dfX = PP.transform(X)

    
    print("═" * 70)
    print("DETALHAMENTO DAS NOVAS COLUNAS CRIADAS")
    print("═" * 70)

# Identificar novas colunas
    original_cols = X.columns.tolist() if hasattr(X, 'columns') else []
    transformed_cols = dfX.columns.tolist() if hasattr(dfX, 'columns') else []
    new_cols = [col for col in transformed_cols if col not in original_cols]

    print(f"\n📈 RESUMO ESTATÍSTICO:")
    print(f"   • Dimensão originais: {X.shape}")
    print(f"   • Colunas originais: {len(original_cols)}")
    print(f"   • Colunas após transformação: {len(transformed_cols)}")
    print(f"   • Novas colunas criadas: {len(new_cols)}")
    print(f"   • Aumento: {(len(new_cols)/len(original_cols)*100):.1f}%")

    if new_cols:
        print(f"\n📋 LISTA DAS NOVAS COLUNAS ({len(new_cols)} no total):")
        print("-" * 50)
        
        # Exibir em colunas para melhor visualização
        for i in range(0, len(new_cols), 3):
            line_cols = new_cols[i:i+3]
            col_str = "   ".join([f"{col:<25}" for col in line_cols])
            print(f"   {col_str}")
        
        print(f"\n🔍 DETALHES DAS NOVAS COLUNAS:")
        print("-" * 50)
        
        for col in new_cols[:len(new_cols)]:
            unique_vals = dfX[col].nunique() if hasattr(dfX[col], 'nunique') else 'N/A'
            dtype = dfX[col].dtype
            non_null = dfX[col].count() if hasattr(dfX[col], 'count') else 'N/A'
            total = len(dfX)
            
            print(f"   • {col} {dtype}")
            print(f"      Valores únicos: {unique_vals}")
            print(f"      Não nulos: {non_null}/{total} ({(non_null/total*100):.1f}%)")
            
            # Mostrar valores únicos se forem poucos
            if hasattr(dfX[col], 'unique') and unique_vals <= 5:
                vals = dfX[col].unique()[:5]
                print(f"     Valores: {vals}")
            print()
        
    else:
        print("\nℹ️  Nenhuma nova coluna criada - transformações aplicadas in-place.")
    
    print(f"\n👁️  VISUALIZAÇÃO (primeiras 5 linhas):")
    display(dfX.head(5))
    return dfX

In [3]:
#X.head(5)

### ⚙️ preprocessador V1.2

In [4]:
# print(f"{'='*70}")
# print(f"{'⚙️ preprocessador V1.2'.center(70)}")
# print(f"{'='*70}")
# BASE = Path.cwd().parent
# PP1 = joblib.load(BASE/'src'/'preprocess_diabetes_v1.2.joblib')['preprocessador']
# display(PP1)
# out1=vizualizar_transformacao(PP1,X)

### ⚙️ preprocessador V1.4

In [5]:
# print(f"{'='*70}")
# print(f"{'⚙️ preprocessador V1.4'.center(70)}")
# print(f"{'='*70}")
# BASE = Path.cwd().parent
# PP2 = joblib.load(BASE/'src'/'preprocess_diabetes_v1.4.joblib')['preprocessador']
# display(PP2)
# out2=vizualizar_transformacao(PP2,X)


### ⚙️ preprocessador V2

In [6]:
# from src.preprocess_utils_diab2 import * #importar uma vez que o codigo possui classes proprias
# print(f"{'='*70}")
# print(f"{'⚙️ preprocessador V2'.center(70)}")
# print(f"{'='*70}")
# PP3 = joblib.load(BASE/'src'/'preprocess_diabetes_v2.joblib')['preprocessador']
# display(PP3)
# out3=vizualizar_transformacao(PP3,X)

### ⚙️ preprocessador V3

In [7]:
from src.preprocess_utils_diab3 import *

print(f"{'='*70}")
print(f"{'⚙️ preprocessador V3'.center(70)}")
print(f"{'='*70}")
PP4 = joblib.load(BASE/'src'/'preprocess_diabetes_v3.joblib')['preprocessador']
display(PP4)
out4=vizualizar_transformacao(PP4,X)

                         ⚙️ preprocessador V3                         


Pipeline(steps=[('ordinal_map', OrdinalMapper()),
                ('feature_engineering', RiskFeatureEngineer()),
                ('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imp',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('float32',
                                                                   FunctionTransformer(func=<function to_float32 at 0x79a58047f5e0>))]),
                                                  Index(['age', 'alcohol_consumption_per_week', 'bmi', 'cholesterol_total'...
                                                                   FunctionTransformer(func=<function to_category at 0x79a58047f0d0>))]),
                                                  Index(['gender', 'ethnicity', 'smoking_status', 'employment_status',
       'Age_Group', 'PAW_Group'],
      dtype='object')),
                                                 ('bin',
                                                  Pipeline(steps=[('int8',
                                                                   FunctionTransformer(func=<function to_int8 at 0x79a58047f670>))]),
                                                  ['family_history_diabetes',
                                                   'hypertension_history',
                                                   'cardiovascular_history'])],
                                   verbose_feature_names_out=False))])

══════════════════════════════════════════════════════════════════════
DETALHAMENTO DAS NOVAS COLUNAS CRIADAS
══════════════════════════════════════════════════════════════════════

📈 RESUMO ESTATÍSTICO:
   • Dimensão originais: (700000, 26)
   • Colunas originais: 26
   • Colunas após transformação: 33
   • Novas colunas criadas: 9
   • Aumento: 34.6%

📋 LISTA DAS NOVAS COLUNAS (9 no total):
--------------------------------------------------
   continuous_risk_score       global_prob_risk            risk_age_cont            
   risk_age_p                  risk_fh_p                   risk_paw_cont            
   risk_paw_p                  Age_Group                   PAW_Group                

🔍 DETALHES DAS NOVAS COLUNAS:
--------------------------------------------------
   • continuous_risk_score float32
      Valores únicos: 30914
      Não nulos: 700000/700000 (100.0%)

   • global_prob_risk float32
      Valores únicos: 48
      Não nulos: 700000/700000 (100.0%)

   • risk_age_co

,age,alcohol_consumption_per_week,bmi,cholesterol_total,continuous_risk_score,diastolic_bp,diet_score,global_prob_risk,hdl_cholesterol,heart_rate,...,income_level,gender,ethnicity,smoking_status,employment_status,Age_Group,PAW_Group,family_history_diabetes,hypertension_history,cardiovascular_history
0,31.0,1.0,33.400002,199.0,0.084485,70.0,7.7,0.588708,58.0,62.0,...,1,Female,Hispanic,Current,Employed,Adulto (30–44),Baixo (<150),0,0,0
1,50.0,2.0,23.799999,199.0,0.173410,77.0,5.7,0.614308,50.0,71.0,...,3,Female,White,Never,Employed,Meia-idade (45–59),Baixo (<150),0,0,0
2,32.0,3.0,24.100000,188.0,0.075634,89.0,8.5,0.511612,59.0,73.0,...,1,Male,Hispanic,Never,Retired,Adulto (30–44),Recomendado (150-300),0,0,0
3,54.0,3.0,26.600000,182.0,0.192804,69.0,4.6,0.614308,54.0,74.0,...,1,Female,White,Current,Employed,Meia-idade (45–59),Baixo (<150),0,1,0
4,54.0,1.0,28.799999,206.0,0.196832,60.0,5.7,0.614308,49.0,85.0,...,3,Male,White,Never,Retired,Meia-idade (45–59),Baixo (<150),0,1,0


### ⚙️ preprocessador V3a

In [8]:
from src.preprocess_utils_diab3a import *

print(f"{'='*70}")
print(f"{'⚙️ preprocessador V3a'.center(70)}")
print(f"{'='*70}")
PP41 = joblib.load(BASE/'src'/'preprocess_diabetes_v3a.joblib')['preprocessador']
out41=PP41.transform(X)
out41=vizualizar_transformacao(PP41,X)

                        ⚙️ preprocessador V3a                         
══════════════════════════════════════════════════════════════════════
DETALHAMENTO DAS NOVAS COLUNAS CRIADAS
══════════════════════════════════════════════════════════════════════

📈 RESUMO ESTATÍSTICO:
   • Dimensão originais: (700000, 26)
   • Colunas originais: 26
   • Colunas após transformação: 33
   • Novas colunas criadas: 9
   • Aumento: 34.6%

📋 LISTA DAS NOVAS COLUNAS (9 no total):
--------------------------------------------------
   continuous_risk_score       global_prob_risk            risk_age_cont            
   risk_age_p                  risk_fh_p                   risk_paw_cont            
   risk_paw_p                  Age_Group                   PAW_Group                

🔍 DETALHES DAS NOVAS COLUNAS:
--------------------------------------------------
   • continuous_risk_score float32
      Valores únicos: 30990
      Não nulos: 700000/700000 (100.0%)

   • global_prob_risk float32
      Valor

,age,alcohol_consumption_per_week,bmi,cholesterol_total,continuous_risk_score,diastolic_bp,diet_score,global_prob_risk,hdl_cholesterol,heart_rate,...,income_level,gender,ethnicity,smoking_status,employment_status,Age_Group,PAW_Group,family_history_diabetes,hypertension_history,cardiovascular_history
0,31.0,1.0,33.400002,199.0,0.602030,70.0,7.7,0.588708,58.0,62.0,...,1,Female,Hispanic,Current,Employed,Adulto (30–44),Baixo (<150),0,0,0
1,50.0,2.0,23.799999,199.0,0.645795,77.0,5.7,0.614308,50.0,71.0,...,3,Female,White,Never,Employed,Meia-idade (45–59),Baixo (<150),0,0,0
2,32.0,3.0,24.100000,188.0,0.509807,89.0,8.5,0.511612,59.0,73.0,...,1,Male,Hispanic,Never,Retired,Adulto (30–44),Recomendado (150-300),0,0,0
3,54.0,3.0,26.600000,182.0,0.656614,69.0,4.6,0.614308,54.0,74.0,...,1,Female,White,Current,Employed,Meia-idade (45–59),Baixo (<150),0,1,0
4,54.0,1.0,28.799999,206.0,0.675261,60.0,5.7,0.614308,49.0,85.0,...,3,Male,White,Never,Retired,Meia-idade (45–59),Baixo (<150),0,1,0


In [9]:
out41.info()
#out41[['education_level','income_level']]
#out41['education_level','income_level'].unique()
out41.select_dtypes(include=['category'])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700000 entries, 0 to 699999
Data columns (total 33 columns):
 #   Column                              Non-Null Count   Dtype   
---  ------                              --------------   -----   
 0   age                                 700000 non-null  float32 
 1   alcohol_consumption_per_week        700000 non-null  float32 
 2   bmi                                 700000 non-null  float32 
 3   cholesterol_total                   700000 non-null  float32 
 4   continuous_risk_score               700000 non-null  float32 
 5   diastolic_bp                        700000 non-null  float32 
 6   diet_score                          700000 non-null  float32 
 7   global_prob_risk                    700000 non-null  float32 
 8   hdl_cholesterol                     700000 non-null  float32 
 9   heart_rate                          700000 non-null  float32 
 10  ldl_cholesterol                     700000 non-null  float32 
 11  physical_acti

,gender,ethnicity,smoking_status,employment_status,Age_Group,PAW_Group
0,Female,Hispanic,Current,Employed,Adulto (30–44),Baixo (<150)
1,Female,White,Never,Employed,Meia-idade (45–59),Baixo (<150)
2,Male,Hispanic,Never,Retired,Adulto (30–44),Recomendado (150-300)
3,Female,White,Current,Employed,Meia-idade (45–59),Baixo (<150)
4,Male,White,Never,Retired,Meia-idade (45–59),Baixo (<150)
...,...,...,...,...,...,...
699995,Female,Hispanic,Former,Employed,Jovem Adulto (18–29),Baixo (<150)
699996,Female,Hispanic,Former,Employed,Meia-idade (45–59),Baixo (<150)
699997,Female,White,Never,Employed,Adulto (30–44),Baixo (<150)
699998,Female,White,Never,Retired,Meia-idade (45–59),Baixo (<150)


In [10]:
print(f"Max PAW Pipeline: {PP41.named_steps['feature_engineering'].paw_min_}")
print(f"Max PAW Pipeline: {PP41.named_steps['feature_engineering'].paw_max_}")

Max PAW Pipeline: 1
Max PAW Pipeline: 705


In [11]:
print(f"✅ LGBM finalizado • {time.strftime('%H:%M:%S')}")


✅ LGBM finalizado • 15:18:29


In [12]:
print(f"✅ Arquivo de submissão salvo • {time.strftime('%d/%m/%Y-%H:%M:%S')}")

✅ Arquivo de submissão salvo • 06/03/2026-15:18:29


In [15]:
out41.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700000 entries, 0 to 699999
Data columns (total 33 columns):
 #   Column                              Non-Null Count   Dtype   
---  ------                              --------------   -----   
 0   age                                 700000 non-null  float32 
 1   alcohol_consumption_per_week        700000 non-null  float32 
 2   bmi                                 700000 non-null  float32 
 3   cholesterol_total                   700000 non-null  float32 
 4   continuous_risk_score               700000 non-null  float32 
 5   diastolic_bp                        700000 non-null  float32 
 6   diet_score                          700000 non-null  float32 
 7   global_prob_risk                    700000 non-null  float32 
 8   hdl_cholesterol                     700000 non-null  float32 
 9   heart_rate                          700000 non-null  float32 
 10  ldl_cholesterol                     700000 non-null  float32 
 11  physical_acti